# SmolVLA Comprehensive Study

This notebook implements three complementary experiments on the SmolVLA model:

1. **Baseline Benchmarking**: Fine-tune + evaluate on LIBERO, MetaWorld, and VLABench
2. **Ablation Study (Pass0)**: Zero `state_proj` output and measure success rate drops
3. **Gradient Analysis**: Measure state encoder gradient contributions using flow matching loss

## Model Details
- **Model**: `lerobot/smolvla_base` (450M parameters)
- **Framework**: LeRobot (HuggingFace)
- **State Encoder**: `state_proj` (Single Linear layer, state → 1 VLM token)
- **Loss Function**: Flow matching MSE on vector fields
- **VLM Backbone**: SmolVLM2 (SigLIP + SmolLM2 first 16 layers)
- **Action Expert**: Flow matching transformer (~100M params)
- **Benchmarks**: LIBERO + MetaWorld + VLABench

## Evaluation Protocol (from paper)
Fine-tuning is required for LIBERO and MetaWorld — matching the paper's reported results:
- **LIBERO**: Fine-tune on `HuggingFaceVLA/libero`, 100k steps, 10 trials/task
  - Paper results: 87.3% average (Spatial/Object/Goal/Long)
- **MetaWorld**: Fine-tune on `lerobot/metaworld_mt50`, 100k steps, 10 trials/task
  - Paper results: 57.3% average (MT50)
- **VLABench**: General policy (zero-shot with language instructions)

## Expected Results
- Baseline: Success rates after fine-tuning, consistent with paper Table 2
- Ablation: Performance drop when state_proj is zeroed (relative to fine-tuned baseline)
- Gradient: Gradient magnitude at state_proj vs. vision/language encoders
- Cross-validation: Gradient ∝ Performance impact?

---
## 1. Setup: Paths and GPU Configuration

In [ ]:
# Setup paths and mount Drive
from google.colab import drive
from pathlib import Path
import os

# Mount Drive
drive.mount('/content/drive')

# Path definitions - Drive (persistent)
WORKSPACE = '/content/drive/MyDrive/smolvla_study'
RESULTS_DIR = Path(f'{WORKSPACE}/Results')
BASELINE_DIR = Path(f'{WORKSPACE}/Results/baseline')
ABLATION_DIR = Path(f'{WORKSPACE}/Results/ablation')
GRADIENT_DIR = Path(f'{WORKSPACE}/Results/gradient')

# Path definitions - Local (ephemeral)
CHECKPOINTS_DIR = Path('/content/checkpoints')
LOGS_DIR = Path('/content/logs')
DATA_DIR = Path('/content/data')

# Create directories
for d in [RESULTS_DIR, BASELINE_DIR, ABLATION_DIR, GRADIENT_DIR, CHECKPOINTS_DIR, LOGS_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Environment variables
os.environ.pop('PYTHONPATH', None)
os.environ['MUJOCO_GL'] = 'egl'   # headless rendering
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"CUDA: {torch.version.cuda}")
print(f"Workspace: {WORKSPACE}")

---
## 2. Install Dependencies

In [ ]:
%%bash
# System dependencies for headless rendering
apt-get install -q -y ffmpeg libgl1-mesa-glx libosmesa6-dev xvfb

# Clone LeRobot and install with SmolVLA extras
if [ ! -d /content/lerobot ]; then
    git clone --depth 1 https://github.com/huggingface/lerobot.git /content/lerobot
fi
pip install -q -e "/content/lerobot[smolvla]" 2>&1 | tail -5

# Additional simulation dependencies
pip install -q gymnasium mujoco 2>&1 | tail -3

echo 'LeRobot + SmolVLA installed'

In [ ]:
%%bash
# Install LIBERO
if [ ! -d /content/LIBERO ]; then
    git clone --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git /content/LIBERO
    pip install -q -e /content/LIBERO 2>&1 | tail -5
fi

# Install MetaWorld
if [ ! -d /content/metaworld ]; then
    git clone --depth 1 https://github.com/Farama-Foundation/Metaworld.git /content/metaworld
    pip install -q -e /content/metaworld 2>&1 | tail -5
fi

# Install VLABench
if [ ! -d /content/VLABench ]; then
    git clone --depth 1 https://github.com/OpenDriveLab/VLABench.git /content/VLABench
    pip install -q -e /content/VLABench 2>&1 | tail -5
fi

echo 'Benchmark environments installed'

In [ ]:
%%bash
# Clone EmbodiedLLM for hook framework
if [ ! -d /content/EmbodiedLLM ]; then
    git clone https://github.com/prithviraj-acharya/EmbodiedLLM.git /content/EmbodiedLLM  # adjust URL
    cd /content/EmbodiedLLM && git checkout AnalyseMultipleHooks
fi
echo 'EmbodiedLLM cloned'

---
# PART 1: BASELINE BENCHMARKING

SmolVLA requires fine-tuning on each benchmark — this matches the paper's evaluation protocol
(Table 2: 87.3% LIBERO avg, 57.3% MetaWorld avg).

Official datasets on HuggingFace Hub:
- **LIBERO**: `HuggingFaceVLA/libero` (preprocessed for LeRobot, 1,693 episodes)
- **MetaWorld**: `lerobot/metaworld_mt50` (2,500 episodes, 50 tasks)
- **VLABench**: general zero-shot policy (no fine-tuning)

Training uses `lerobot-train` and evaluation uses `lerobot-eval` — LeRobot's native CLI.

## 3. Fine-tune and Evaluate on LIBERO

In [ ]:

# LIBERO baseline: fine-tune SmolVLA then evaluate via lerobot native CLI
# Dataset: HuggingFaceVLA/libero (preprocessed, LeRobot-compatible)
# Protocol: 100k steps, batch 4 (paper uses 64 on A100s), 10 trials/task

import subprocess, json, os
from pathlib import Path

LIBERO_SUITES = ['libero_spatial', 'libero_object', 'libero_goal', 'libero_10']
LIBERO_RESULTS = {}
LIBERO_DATASET = 'HuggingFaceVLA/libero'
LIBERO_CKPT = str(CHECKPOINTS_DIR / 'smolvla_libero')
LIBERO_TRAIN_STEPS = 100_000
LIBERO_BATCH = 4   # increase to 64 on multi-GPU

# Install LIBERO extras
subprocess.run(['pip', 'install', '-q', '-e', '/content/lerobot[libero]'], check=True)
print('LIBERO extras installed')


def train_libero():
    """Fine-tune SmolVLA on the full LIBERO dataset."""
    ckpt = Path(LIBERO_CKPT)
    if (ckpt / 'config.json').exists():
        print(f'  Checkpoint already exists at {LIBERO_CKPT}, skipping training')
        return

    print(f'  Fine-tuning on {LIBERO_DATASET} ({LIBERO_TRAIN_STEPS} steps)...')
    cmd = [
        'lerobot-train',
        '--policy.type=smolvla',
        '--policy.load_vlm_weights=true',
        f'--dataset.repo_id={LIBERO_DATASET}',
        f'--output_dir={LIBERO_CKPT}',
        f'--steps={LIBERO_TRAIN_STEPS}',
        f'--batch_size={LIBERO_BATCH}',
        '--policy.device=cuda',
        '--policy.dtype=bfloat16',
        '--policy.gradient_checkpointing=true',
        '--env.type=libero',
        '--eval_freq=10000',
        '--eval.n_episodes=10',
        '--eval.batch_size=1',
        '--job_name=smolvla_libero_baseline',
        f'--wandb.enable=false',
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, cwd='/content/lerobot')
    if result.returncode != 0:
        print('  Training failed:')
        print(result.stderr[-3000:])
    else:
        print('  Training complete')


def eval_libero(suite: str) -> dict:
    """Evaluate fine-tuned SmolVLA on a LIBERO suite (10 episodes/task)."""
    result_file = BASELINE_DIR / f'libero_{suite}.json'
    if result_file.exists():
        with open(result_file) as f:
            return json.load(f)

    eval_log_dir = LOGS_DIR / f'libero_{suite}'
    cmd = [
        'lerobot-eval',
        f'--policy.path={LIBERO_CKPT}',
        '--env.type=libero',
        f'--env.task={suite}',
        '--eval.batch_size=1',
        '--eval.n_episodes=100',   # 10 tasks × 10 episodes
        f'--output_dir={eval_log_dir}',
    ]
    print(f'  Evaluating on {suite}...')
    result = subprocess.run(cmd, capture_output=True, text=True, cwd='/content/lerobot')

    # Parse success rate from eval output
    success_rate = float('nan')
    for line in (result.stdout + result.stderr).splitlines():
        if 'success' in line.lower() and any(c.isdigit() for c in line):
            import re
            nums = re.findall(r'\d+\.?\d*', line)
            if nums:
                val = float(nums[0])
                success_rate = val / 100 if val > 1 else val
                break

    # Also check eval JSON if written
    eval_json = eval_log_dir / 'eval_info.json'
    if eval_json.exists():
        with open(eval_json) as f:
            eval_data = json.load(f)
        success_rate = eval_data.get('success_rate', eval_data.get('avg_sum_rewards', success_rate))

    res = {
        'suite': suite, 'success_rate': success_rate,
        'model': LIBERO_CKPT, 'condition': 'baseline',
        'paper_reference': 87.3,  # paper avg across 4 suites
    }
    with open(result_file, 'w') as f:
        json.dump(res, f, indent=2)
    return res


# Train once, eval per suite
print('=== LIBERO Baseline ===')
train_libero()

for suite in LIBERO_SUITES:
    print(f'\nEvaluating: {suite}')
    LIBERO_RESULTS[suite] = eval_libero(suite)
    print(f'  Success rate: {LIBERO_RESULTS[suite]["success_rate"]:.2%}')

print('\n=== LIBERO Baseline Summary ===')
rates = [v['success_rate'] for v in LIBERO_RESULTS.values()
         if not __import__('math').isnan(v['success_rate'])]
for suite, res in LIBERO_RESULTS.items():
    print(f'  {suite}: {res["success_rate"]:.2%}')
if rates:
    import numpy as np
    print(f'  Average: {np.mean(rates):.2%}  (paper: 87.3%)')


## 4. Baseline Evaluation on MetaWorld

In [ ]:

# MetaWorld baseline: fine-tune SmolVLA then evaluate via lerobot native CLI
# Dataset: lerobot/metaworld_mt50 (2,500 episodes, 50 tasks MT50)
# Protocol: 100k steps, batch 4, 10 trials/task

import subprocess, json, numpy as np
from pathlib import Path

MW_DATASET = 'lerobot/metaworld_mt50'
MW_CKPT = str(CHECKPOINTS_DIR / 'smolvla_metaworld')
MW_TRAIN_STEPS = 100_000
MW_BATCH = 4
MW_RESULTS = {}


def train_metaworld():
    """Fine-tune SmolVLA on MetaWorld MT50."""
    ckpt = Path(MW_CKPT)
    if (ckpt / 'config.json').exists():
        print(f'  Checkpoint exists at {MW_CKPT}, skipping training')
        return

    print(f'  Fine-tuning on {MW_DATASET} ({MW_TRAIN_STEPS} steps)...')
    cmd = [
        'lerobot-train',
        '--policy.type=smolvla',
        '--policy.load_vlm_weights=true',
        f'--dataset.repo_id={MW_DATASET}',
        f'--output_dir={MW_CKPT}',
        f'--steps={MW_TRAIN_STEPS}',
        f'--batch_size={MW_BATCH}',
        '--policy.device=cuda',
        '--policy.dtype=bfloat16',
        '--policy.gradient_checkpointing=true',
        '--job_name=smolvla_metaworld_baseline',
        '--wandb.enable=false',
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, cwd='/content/lerobot')
    if result.returncode != 0:
        print('  Training failed:')
        print(result.stderr[-3000:])
    else:
        print('  Training complete')


def eval_metaworld() -> dict:
    """Evaluate fine-tuned SmolVLA on MetaWorld MT50 (10 episodes × 50 tasks = 500 total)."""
    result_file = BASELINE_DIR / 'metaworld.json'
    if result_file.exists():
        with open(result_file) as f:
            return json.load(f)

    eval_log_dir = LOGS_DIR / 'metaworld'
    cmd = [
        'lerobot-eval',
        f'--policy.path={MW_CKPT}',
        '--env.type=metaworld',
        '--eval.batch_size=1',
        '--eval.n_episodes=500',   # 50 tasks × 10 episodes
        f'--output_dir={eval_log_dir}',
    ]
    print('  Evaluating on MetaWorld MT50...')
    result = subprocess.run(cmd, capture_output=True, text=True, cwd='/content/lerobot')

    success_rate = float('nan')
    for line in (result.stdout + result.stderr).splitlines():
        if 'success' in line.lower() and any(c.isdigit() for c in line):
            import re
            nums = re.findall(r'\d+\.?\d*', line)
            if nums:
                val = float(nums[0])
                success_rate = val / 100 if val > 1 else val
                break

    eval_json = eval_log_dir / 'eval_info.json'
    if eval_json.exists():
        with open(eval_json) as f:
            eval_data = json.load(f)
        success_rate = eval_data.get('success_rate', success_rate)

    res = {
        'dataset': MW_DATASET, 'success_rate': success_rate,
        'model': MW_CKPT, 'condition': 'baseline',
        'paper_reference': 57.3,  # paper MT50 average
    }
    with open(result_file, 'w') as f:
        json.dump(res, f, indent=2)
    return res


print('=== MetaWorld Baseline ===')
train_metaworld()

result_file = BASELINE_DIR / 'metaworld.json'
if result_file.exists():
    with open(result_file) as f:
        MW_RESULTS = json.load(f)
else:
    MW_RESULTS = eval_metaworld()

sr = MW_RESULTS.get('success_rate', float('nan'))
print(f'\nMetaWorld MT50 success rate: {sr:.2%}  (paper: 57.3%)')


## 5. Baseline Evaluation on VLABench (General Policy)

VLABench evaluates language-conditioned manipulation policies.
Since SmolVLA's base model is not specifically tuned for VLABench,
we use a **general policy** approach: evaluate the base model directly
on VLABench tasks using its natural language instruction following capability.

In [ ]:
# VLABench baseline: general SmolVLA policy (zero-shot)

import numpy as np
import torch
import json

VLABENCH_RESULTS = {}
result_file = BASELINE_DIR / 'vlabench.json'

def evaluate_vlabench_category(category: str, policy, preprocess, postprocess,
                                n_episodes: int = 20) -> float:
    """Evaluate SmolVLA on a VLABench category (general policy, zero-shot)."""
    try:
        import vlabench  # VLABench package
        env = vlabench.make(category, render_mode='rgb_array')
    except Exception as e:
        # Fallback: try OpenAI gym / gymnasium registration
        try:
            import gymnasium as gym
            env = gym.make(f'VLABench/{category}-v0', render_mode='rgb_array')
        except Exception as e2:
            print(f'  VLABench {category}: env creation failed ({e2})')
            return float('nan')

    device = next(policy.parameters()).device
    successes = []

    for ep in range(n_episodes):
        obs, info = env.reset()
        policy.reset()
        task_instruction = info.get('task', 'complete the task')
        success = False

        for step in range(300):
            # Extract image and state from observation
            if isinstance(obs, dict):
                img = obs.get('image', obs.get('rgb', np.zeros((128, 128, 3), dtype=np.uint8)))
                state = obs.get('state', obs.get('qpos', np.zeros(7)))
            else:
                img = np.zeros((128, 128, 3), dtype=np.uint8)
                state = obs[:7] if len(obs) >= 7 else np.zeros(7)

            frame = {
                'observation.images.top': img,
                'observation.state': state,
                'task': task_instruction,
            }
            batch = preprocess(frame)
            with torch.inference_mode():
                action = policy.select_action(batch)
                action = postprocess(action)
            obs, reward, terminated, truncated, info = env.step(action.cpu().numpy())
            if info.get('success', False) or reward > 0.9:
                success = True
                break
            if terminated or truncated:
                success = info.get('success', False)
                break
        successes.append(float(success))
    env.close()
    return float(np.mean(successes))


VLABENCH_CATEGORIES = [
    'pick_place', 'stack', 'pour', 'insert', 'open_close'
]

print('Running VLABench baseline (general SmolVLA policy)...')

if result_file.exists():
    print('  Loading cached results')
    with open(result_file) as f:
        VLABENCH_RESULTS = json.load(f)
else:
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    from lerobot.policies.factory import make_pre_post_processors

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    policy = SmolVLAPolicy.from_pretrained('lerobot/smolvla_base').to(device).eval()
    preprocess, postprocess = make_pre_post_processors(
        policy.config, 'lerobot/smolvla_base',
        preprocessor_overrides={'device_processor': {'device': str(device)}}
    )

    for cat in VLABENCH_CATEGORIES:
        print(f'  Category: {cat}')
        sr = evaluate_vlabench_category(cat, policy, preprocess, postprocess)
        VLABENCH_RESULTS[cat] = {
            'success_rate': sr,
            'model': 'lerobot/smolvla_base',
            'condition': 'baseline_general',
        }
        print(f'    Success rate: {sr:.2%}')

    with open(result_file, 'w') as f:
        json.dump(VLABENCH_RESULTS, f, indent=2)

print('\n=== VLABench Baseline Summary (General Policy) ===')
rates = [v['success_rate'] for v in VLABENCH_RESULTS.values()
         if not np.isnan(v['success_rate'])]
for cat, res in VLABENCH_RESULTS.items():
    print(f'  {cat}: {res["success_rate"]:.2%}')
if rates:
    print(f'  Average: {np.mean(rates):.2%}')

---
# PART 2: ABLATION STUDY (Pass0 — Zero State Encoder)

We zero out the `state_proj` output using a forward hook,
rerun evaluation on all benchmarks, and measure the success rate drop.

## 6. Create Ablated SmolVLA Wrapper

In [ ]:
# Ablated SmolVLA: register forward hook to zero state_proj output

import torch
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors

ABLATION_HANDLE = None  # Track the registered hook


def load_smolvla(checkpoint: str = 'lerobot/smolvla_base',
                 device: str = 'cuda') -> tuple:
    """Load SmolVLA and create preprocessor/postprocessor."""
    _device = torch.device(device if torch.cuda.is_available() else 'cpu')
    policy = SmolVLAPolicy.from_pretrained(checkpoint).to(_device).eval()
    preprocess, postprocess = make_pre_post_processors(
        policy.config, checkpoint,
        preprocessor_overrides={'device_processor': {'device': str(_device)}}
    )
    return policy, preprocess, postprocess


def find_state_proj(policy) -> torch.nn.Module:
    """
    Locate state_proj (linear layer) inside SmolVLA.
    Tries multiple attribute paths used across lerobot versions.
    """
    # Unwrap SmolVLAPolicy -> inner model
    inner = getattr(policy, 'model', policy)

    # Direct attribute search
    for attr in ['state_proj', 'proprio_proj', 'state_token_proj', 'state_encoder']:
        if hasattr(inner, attr):
            module = getattr(inner, attr)
            print(f'  Found state encoder at policy.model.{attr}: {module}')
            return module

    # Fallback: named_modules scan
    for name, module in policy.named_modules():
        if 'state' in name.lower() and isinstance(module, torch.nn.Linear):
            print(f'  Found state encoder via scan at {name}: {module}')
            return module

    raise RuntimeError('Could not find state_proj in SmolVLA. '
                       'Run policy.named_modules() to inspect manually.')


def attach_zero_state_hook(policy) -> torch.utils.hooks.RemovableHook:
    """
    Register a forward hook that zeros out the state_proj output.
    This is the Pass0 ablation: remove proprioceptive information entirely.
    """
    state_proj = find_state_proj(policy)

    def zero_output(module, input, output):
        return torch.zeros_like(output)

    handle = state_proj.register_forward_hook(zero_output)
    print(f'  Zero-state hook registered on {state_proj.__class__.__name__}')
    return handle


# Quick test
print('Testing state_proj discovery...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
test_policy, _, _ = load_smolvla(device=device)
state_proj_module = find_state_proj(test_policy)
print(f'  state_proj: {state_proj_module}')
del test_policy

## 7. Run Ablation on LIBERO (zero state_proj, fine-tuned model)

In [ ]:

# LIBERO ablation: run lerobot-eval with a zero-state-proj wrapper checkpoint
# Strategy: patch the fine-tuned checkpoint's state_proj in-memory, save to temp dir,
# then run lerobot-eval against the patched checkpoint.

import json, shutil, torch, numpy as np
from pathlib import Path

LIBERO_ABLATION = {}
ABLATED_LIBERO_CKPT = str(CHECKPOINTS_DIR / 'smolvla_libero_ablated')


def create_ablated_checkpoint(src_ckpt: str, dst_ckpt: str):
    """
    Copy checkpoint and zero out state_proj weights so lerobot-eval
    uses a model where state is fully ignored.
    """
    src = Path(src_ckpt)
    dst = Path(dst_ckpt)
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

    # Zero state_proj in the saved model weights
    import safetensors.torch as st
    weight_file = dst / 'model.safetensors'
    if not weight_file.exists():
        # Try pytorch_model.bin
        weight_file = dst / 'pytorch_model.bin'

    if weight_file.suffix == '.safetensors':
        weights = st.load_file(weight_file)
        for k in list(weights.keys()):
            if 'state_proj' in k:
                weights[k] = torch.zeros_like(weights[k])
                print(f'  Zeroed: {k}')
        st.save_file(weights, weight_file)
    else:
        weights = torch.load(weight_file, map_location='cpu')
        state_dict = weights.get('model', weights)
        for k in list(state_dict.keys()):
            if 'state_proj' in k:
                state_dict[k] = torch.zeros_like(state_dict[k])
                print(f'  Zeroed: {k}')
        torch.save(weights, weight_file)

    print(f'  Ablated checkpoint saved to {dst_ckpt}')


import subprocess

# Create ablated checkpoint once
if not (Path(ABLATED_LIBERO_CKPT) / 'config.json').exists():
    src = Path(LIBERO_CKPT)
    if not (src / 'config.json').exists():
        print('WARNING: Fine-tuned LIBERO checkpoint not found. Run cell 3 first.')
    else:
        print('Creating ablated checkpoint (zeroing state_proj)...')
        create_ablated_checkpoint(LIBERO_CKPT, ABLATED_LIBERO_CKPT)
else:
    print(f'Ablated checkpoint ready: {ABLATED_LIBERO_CKPT}')


def eval_libero_ablated(suite: str) -> dict:
    """Evaluate ablated SmolVLA on a LIBERO suite."""
    result_file = ABLATION_DIR / f'libero_{suite}.json'
    if result_file.exists():
        with open(result_file) as f:
            return json.load(f)

    eval_log_dir = LOGS_DIR / f'libero_ablated_{suite}'
    cmd = [
        'lerobot-eval',
        f'--policy.path={ABLATED_LIBERO_CKPT}',
        '--env.type=libero',
        f'--env.task={suite}',
        '--eval.batch_size=1',
        '--eval.n_episodes=100',
        f'--output_dir={eval_log_dir}',
    ]
    print(f'  Evaluating ablated model on {suite}...')
    result = subprocess.run(cmd, capture_output=True, text=True, cwd='/content/lerobot')

    success_rate = float('nan')
    for line in (result.stdout + result.stderr).splitlines():
        if 'success' in line.lower() and any(c.isdigit() for c in line):
            import re
            nums = re.findall(r'\d+\.?\d*', line)
            if nums:
                val = float(nums[0])
                success_rate = val / 100 if val > 1 else val
                break

    eval_json = eval_log_dir / 'eval_info.json'
    if eval_json.exists():
        with open(eval_json) as f:
            eval_data = json.load(f)
        success_rate = eval_data.get('success_rate', success_rate)

    base_sr = LIBERO_RESULTS.get(suite, {}).get('success_rate', float('nan'))
    res = {
        'suite': suite, 'success_rate': success_rate,
        'baseline': base_sr, 'condition': 'ablation_zero_state',
    }
    with open(result_file, 'w') as f:
        json.dump(res, f, indent=2)
    return res


print('\n=== LIBERO Ablation (zero state_proj) ===')
for suite in LIBERO_SUITES:
    LIBERO_ABLATION[suite] = eval_libero_ablated(suite)
    base = LIBERO_ABLATION[suite].get('baseline', float('nan'))
    ablated = LIBERO_ABLATION[suite]['success_rate']
    drop = base - ablated if not np.isnan(base) else float('nan')
    print(f'  {suite}: baseline={base:.2%}  ablated={ablated:.2%}  drop={drop:.2%}')


## 8. Run Ablation on MetaWorld and VLABench

In [ ]:

# MetaWorld + VLABench ablation

import json, shutil, torch, numpy as np, subprocess
from pathlib import Path

MW_ABLATION = {}
VLABENCH_ABLATION = {}
ABLATED_MW_CKPT = str(CHECKPOINTS_DIR / 'smolvla_metaworld_ablated')

# --- MetaWorld: create ablated checkpoint from fine-tuned MW model ---
mw_ablation_file = ABLATION_DIR / 'metaworld.json'

if not mw_ablation_file.exists():
    mw_src = Path(MW_CKPT)
    if not (mw_src / 'config.json').exists():
        print('WARNING: Fine-tuned MetaWorld checkpoint not found. Run cell 4 first.')
    else:
        if not (Path(ABLATED_MW_CKPT) / 'config.json').exists():
            print('Creating ablated MetaWorld checkpoint...')
            create_ablated_checkpoint(MW_CKPT, ABLATED_MW_CKPT)

        eval_log_dir = LOGS_DIR / 'metaworld_ablated'
        cmd = [
            'lerobot-eval',
            f'--policy.path={ABLATED_MW_CKPT}',
            '--env.type=metaworld',
            '--eval.batch_size=1',
            '--eval.n_episodes=500',
            f'--output_dir={eval_log_dir}',
        ]
        print('  Evaluating ablated model on MetaWorld MT50...')
        result = subprocess.run(cmd, capture_output=True, text=True, cwd='/content/lerobot')

        success_rate = float('nan')
        for line in (result.stdout + result.stderr).splitlines():
            if 'success' in line.lower() and any(c.isdigit() for c in line):
                import re
                nums = re.findall(r'\d+\.?\d*', line)
                if nums:
                    val = float(nums[0])
                    success_rate = val / 100 if val > 1 else val
                    break

        eval_json = eval_log_dir / 'eval_info.json'
        if eval_json.exists():
            with open(eval_json) as f:
                eval_data = json.load(f)
            success_rate = eval_data.get('success_rate', success_rate)

        base_sr = MW_RESULTS.get('success_rate', float('nan'))
        MW_ABLATION = {
            'success_rate': success_rate, 'baseline': base_sr,
            'condition': 'ablation_zero_state',
        }
        with open(mw_ablation_file, 'w') as f:
            json.dump(MW_ABLATION, f, indent=2)
else:
    with open(mw_ablation_file) as f:
        MW_ABLATION = json.load(f)

# --- VLABench ablation (zero-shot, use base model + hook) ---
vlab_ablation_file = ABLATION_DIR / 'vlabench.json'

if not vlab_ablation_file.exists():
    policy, preprocess, postprocess = load_smolvla()
    handle = attach_zero_state_hook(policy)
    try:
        for cat in VLABENCH_CATEGORIES:
            print(f'  VLABench ablation: {cat}')
            sr = evaluate_vlabench_category(cat, policy, preprocess, postprocess)
            base_sr = VLABENCH_RESULTS.get(cat, {}).get('success_rate', float('nan'))
            VLABENCH_ABLATION[cat] = {
                'success_rate': sr, 'baseline': base_sr,
                'condition': 'ablation_zero_state',
            }
            drop = (base_sr - sr) if not np.isnan(base_sr) else float('nan')
            print(f'    Baseline: {base_sr:.2%} | Ablated: {sr:.2%} | Drop: {drop:.2%}')
    finally:
        handle.remove()
    with open(vlab_ablation_file, 'w') as f:
        json.dump(VLABENCH_ABLATION, f, indent=2)
else:
    with open(vlab_ablation_file) as f:
        VLABENCH_ABLATION = json.load(f)

print('\n=== Ablation Summary (state_proj zeroed) ===')
mw_base = MW_ABLATION.get('baseline', float('nan'))
mw_abl = MW_ABLATION.get('success_rate', float('nan'))
print(f'MetaWorld MT50: baseline={mw_base:.2%}  ablated={mw_abl:.2%}  drop={(mw_base-mw_abl):.2%}')
print('VLABench:')
for cat, res in VLABENCH_ABLATION.items():
    drop = res.get('baseline', float('nan')) - res['success_rate']
    print(f'  {cat}: {res["success_rate"]:.2%} (drop: {drop:.2%})')


---
# PART 3: GRADIENT ANALYSIS

We measure how much gradient flows through `state_proj` compared to
the vision and language encoders during flow matching training.

## 9. Environment Setup

In [ ]:
# Check GPU
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Running on CPU')

In [ ]:
# Install pip dependencies for analysis
import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'torch', 'torchvision', 'transformers', 'accelerate',
    'matplotlib', 'seaborn', 'scikit-learn', 'numpy', 'Pillow'
], check=True)
print('Dependencies ready')

In [ ]:
# Add EmbodiedLLM to path and import SmolVLAHooks
import sys
sys.path.insert(0, '/content/EmbodiedLLM/MultipleHooksStudy')

import torch
import torch.nn as nn
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from hooks.model_specific.smolvla_hooks import SmolVLAHooks
print('SmolVLAHooks imported successfully')

## 10. Load SmolVLA Model

In [ ]:
# Load SmolVLA via lerobot
import torch
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_ID = 'lerobot/smolvla_base'

print(f'Loading {MODEL_ID}...')
policy = SmolVLAPolicy.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
).to(device)

n_params = sum(p.numel() for p in policy.parameters())
print(f'Loaded SmolVLA: {n_params/1e6:.0f}M parameters')
print(f'Device: {device}')

## 11. Discover Model Structure

In [ ]:
# Initialize hook manager and discover SmolVLA structure
from hooks.model_specific.smolvla_hooks import SmolVLAHooks

hook_manager = SmolVLAHooks(policy)
structure = hook_manager.discover_model_structure()

print('=== SmolVLA Model Structure ===')
print(f'Model name: {structure["model_name"]}')
print(f'Has state encoder: {structure["has_proprio_encoder"]}')
print(f'State encoder type: {structure["proprio_encoder_type"]}')
if 'state_input_dim' in structure:
    print(f'state_proj dims: {structure["state_input_dim"]} → {structure["state_output_dim"]}')
print('\nDiscovered components:')
for k, v in structure.get('components', {}).items():
    print(f'  {k}: {v}')

# Also print named modules to verify
print('\nTop-level module names:')
for name, module in policy.named_children():
    print(f'  {name}: {module.__class__.__name__}')

In [ ]:
# Deep dive into inner model structure
inner = getattr(policy, 'model', policy)
print('Inner model top-level:')
for name, module in inner.named_children():
    n_params = sum(p.numel() for p in module.parameters())
    print(f'  {name}: {module.__class__.__name__} ({n_params/1e6:.1f}M params)')

## 12. Attach Analysis Hooks

In [ ]:
# Attach gradient hooks
hook_manager.attach_gradient_hooks()
print('Gradient hooks attached')

# Print what we're monitoring
print(f'  Monitoring state_proj: {hook_manager.state_proj is not None}')
print(f'  Monitoring vision_encoder: {hook_manager.vision_encoder is not None}')
print(f'  Monitoring language_encoder: {hook_manager.language_encoder is not None}')
print(f'  Monitoring action_expert layers: {hook_manager.transformer_layers is not None}')

## 13. Prepare Sample Data

In [ ]:
# Create synthetic batch for gradient analysis
# SmolVLA config: 2 cameras, 128x128 images, 7-DOF state, 50-step action chunks

import torch
from PIL import Image
import numpy as np

BATCH_SIZE = 2
IMG_SIZE = 128
STATE_DIM = 7      # 7-DOF robot arm
ACTION_DIM = 7     # 7-DOF actions
CHUNK_SIZE = 50    # SmolVLA action chunk length

# Try loading from lerobot dataset if available
USE_REAL_DATA = False
try:
    from lerobot.datasets.lerobot_dataset import LeRobotDataset
    # Try a small public dataset
    ds = LeRobotDataset('lerobot/aloha_sim_insertion_scripted', split='train')
    sample = dict(ds[0])
    print(f'Using real dataset: {ds.repo_id}')
    print(f'Sample keys: {list(sample.keys())}')
    USE_REAL_DATA = True
except Exception as e:
    print(f'No real dataset available ({e}), using synthetic data')


def make_synthetic_batch(batch_size=BATCH_SIZE, device='cpu'):
    """Create a synthetic batch compatible with SmolVLA forward pass."""
    dtype = torch.bfloat16 if device != 'cpu' else torch.float32
    # Images: [B, C, H, W] normalized
    img = torch.randn(batch_size, 3, IMG_SIZE, IMG_SIZE, dtype=dtype, device=device)
    # Robot state
    state = torch.randn(batch_size, STATE_DIM, dtype=dtype, device=device)
    # Target actions (for flow matching loss)
    actions = torch.randn(batch_size, CHUNK_SIZE, ACTION_DIM, dtype=dtype, device=device)

    batch = {
        # Image keys (actual keys depend on policy.config.input_features)
        'observation.images.top': img,
        'observation.images.wrist': img.clone(),
        'observation.state': state,
        'action': actions,
        'action_is_pad': torch.zeros(batch_size, CHUNK_SIZE, dtype=torch.bool, device=device),
    }
    return batch


# Discover actual input keys from policy config
print('\nPolicy input features:')
if hasattr(policy, 'config') and hasattr(policy.config, 'input_features'):
    for k, v in policy.config.input_features.items():
        print(f'  {k}: {v}')
else:
    print('  (config not available, using synthetic defaults)')

print(f'\nSynthetic batch shape: images={IMG_SIZE}x{IMG_SIZE}, state={STATE_DIM}D, actions={CHUNK_SIZE}x{ACTION_DIM}D')

## 14. Baseline Forward + Backward Pass

In [ ]:
# Baseline: run forward/backward with normal state encoding

import torch

policy.train()  # Enable gradients
device = next(policy.parameters()).device

# Build batch using correct input feature keys
if hasattr(policy, 'config') and hasattr(policy.config, 'input_features'):
    img_keys = [k for k in policy.config.input_features if 'image' in k.lower()]
    state_key = next((k for k in policy.config.input_features if 'state' in k.lower()), None)
else:
    img_keys = ['observation.images.top']
    state_key = 'observation.state'

batch = make_synthetic_batch(device=str(device))

# Ensure all required keys are present
for k in img_keys:
    if k not in batch:
        batch[k] = batch.get('observation.images.top',
                              torch.randn(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE,
                                          dtype=torch.bfloat16, device=device))

# Forward pass: SmolVLA policy.forward returns (loss, outputs)
try:
    loss, outputs = policy.forward(batch)
    print(f'Forward pass successful')
    print(f'  Flow matching loss: {loss.item():.4f}')
except Exception as e:
    print(f'Forward pass failed: {e}')
    # Try alternative: policy.compute_loss(batch)
    try:
        loss = policy.compute_loss(batch)
        if isinstance(loss, tuple):
            loss = loss[0]
        print(f'  Flow matching loss (compute_loss): {loss.item():.4f}')
    except Exception as e2:
        print(f'  compute_loss also failed: {e2}')
        raise

# Backward pass
policy.zero_grad()
loss.backward()
print(f'Backward pass successful')

# Extract baseline gradient info
baseline_grads = {}
for name, param in policy.named_parameters():
    if param.grad is not None:
        baseline_grads[name] = param.grad.detach().float().norm().item()

print(f'\nGradients computed for {len(baseline_grads)} parameters')

## 15. Baseline Gradient Statistics

In [ ]:
# Compute gradient statistics for each component

import numpy as np

def get_module_gradient_norm(module: torch.nn.Module) -> float:
    """Compute total gradient norm for a module's parameters."""
    total_norm = 0.0
    count = 0
    for param in module.parameters():
        if param.grad is not None:
            total_norm += param.grad.detach().float().norm().item() ** 2
            count += 1
    return float(np.sqrt(total_norm)) if count > 0 else 0.0


# Gather gradient norms for each encoder
baseline_stats = {}

if hook_manager.state_proj is not None:
    baseline_stats['state_proj'] = get_module_gradient_norm(hook_manager.state_proj)

if hook_manager.vision_encoder is not None:
    baseline_stats['vision_encoder'] = get_module_gradient_norm(hook_manager.vision_encoder)

if hook_manager.language_encoder is not None:
    baseline_stats['language_encoder'] = get_module_gradient_norm(hook_manager.language_encoder)

if hook_manager.action_expert is not None:
    baseline_stats['action_expert'] = get_module_gradient_norm(hook_manager.action_expert)

print('=== Baseline Gradient Norms ===')
for name, norm in sorted(baseline_stats.items(), key=lambda x: -x[1]):
    print(f'  {name}: {norm:.6f}')

# Compute ratios relative to vision encoder
if 'vision_encoder' in baseline_stats and baseline_stats['vision_encoder'] > 0:
    print('\nGradient Ratios (relative to vision_encoder):')
    for name, norm in baseline_stats.items():
        ratio = norm / baseline_stats['vision_encoder']
        print(f'  {name}/vision_encoder: {ratio:.4f}')

# Check state_proj weight and bias gradients directly
if hook_manager.state_proj is not None:
    sp = hook_manager.state_proj
    if sp.weight.grad is not None:
        print(f'\nstate_proj weight gradient:')
        print(f'  norm: {sp.weight.grad.float().norm().item():.6f}')
        print(f'  mean: {sp.weight.grad.float().mean().item():.6f}')
        print(f'  max:  {sp.weight.grad.float().abs().max().item():.6f}')

## 16. Ablation: Zero Out State Encoder Output

In [ ]:
# Ablation: register hook to zero state_proj, rerun forward+backward

# Register zero-output hook on state_proj
state_proj_module = find_state_proj(policy)

ablation_activations = {}

def zero_and_capture(module, input, output):
    """Zero out state_proj output and capture original for diagnostics."""
    ablation_activations['state_proj_original'] = output.detach().clone()
    return torch.zeros_like(output)

ablation_handle = state_proj_module.register_forward_hook(zero_and_capture)
print(f'Ablation hook registered on: {state_proj_module.__class__.__name__}')

# Forward + backward with ablated state
policy.zero_grad()
try:
    ablated_loss, ablated_outputs = policy.forward(batch)
except Exception:
    ablated_loss = policy.compute_loss(batch)
    if isinstance(ablated_loss, tuple):
        ablated_loss = ablated_loss[0]

ablated_loss.backward()
ablation_handle.remove()  # Always remove after use

print(f'Ablated forward pass:')
print(f'  Baseline loss:  {loss.item():.4f}')
print(f'  Ablated loss:   {ablated_loss.item():.4f}')
print(f'  Loss increase:  {ablated_loss.item() - loss.item():.4f}')
if 'state_proj_original' in ablation_activations:
    orig = ablation_activations['state_proj_original']
    print(f'  Original state_proj output norm: {orig.float().norm().item():.4f}')

## 17. Compare Baseline vs. Ablated Gradients

In [ ]:
# Compare gradient norms: baseline vs ablated

ablated_stats = {}

if hook_manager.state_proj is not None:
    ablated_stats['state_proj'] = get_module_gradient_norm(hook_manager.state_proj)
if hook_manager.vision_encoder is not None:
    ablated_stats['vision_encoder'] = get_module_gradient_norm(hook_manager.vision_encoder)
if hook_manager.language_encoder is not None:
    ablated_stats['language_encoder'] = get_module_gradient_norm(hook_manager.language_encoder)
if hook_manager.action_expert is not None:
    ablated_stats['action_expert'] = get_module_gradient_norm(hook_manager.action_expert)

print('=== Gradient Comparison: Baseline vs Ablated ===')
print(f'{"Module":<25} {"Baseline":>12} {"Ablated":>12} {"Rel. Change":>12}')
print('-' * 65)

comparison = {}
for name in sorted(set(list(baseline_stats.keys()) + list(ablated_stats.keys()))):
    b = baseline_stats.get(name, 0.0)
    a = ablated_stats.get(name, 0.0)
    rel_change = (a - b) / (b + 1e-10) * 100
    comparison[name] = {
        'baseline': b, 'ablated': a, 'relative_change_pct': rel_change
    }
    flag = '**' if abs(rel_change) > 20 else ''
    print(f'{name:<25} {b:>12.6f} {a:>12.6f} {rel_change:>+11.1f}% {flag}')

print('\n** = Large gradient change (>20%) when state zeroed')
print(f'\nConclusion:')
sp_change = comparison.get('state_proj', {}).get('relative_change_pct', 0)
if abs(sp_change) > 20:
    print(f'  state_proj gradient changed by {sp_change:+.1f}% → HIGH sensitivity to state')
else:
    print(f'  state_proj gradient changed by {sp_change:+.1f}% → LOW sensitivity to state')

## 18. Visualization: Gradient Sensitivity

In [ ]:
# Gradient visualization

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('SmolVLA: Proprioceptive State Utilization (Gradient Analysis)', fontsize=13)

modules = list(comparison.keys())
x = np.arange(len(modules))
width = 0.35

# --- Plot 1: Gradient Norms ---
ax = axes[0]
b_vals = [comparison[m]['baseline'] for m in modules]
a_vals = [comparison[m]['ablated'] for m in modules]
bars1 = ax.bar(x - width/2, b_vals, width, label='Baseline', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, a_vals, width, label='State Ablated', color='coral', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(modules, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Gradient Norm')
ax.set_title('Gradient Norms by Module')
ax.legend()
ax.set_yscale('log')

# --- Plot 2: Relative Change ---
ax = axes[1]
rel_changes = [comparison[m]['relative_change_pct'] for m in modules]
colors = ['red' if abs(v) > 20 else 'gray' for v in rel_changes]
bars = ax.bar(modules, rel_changes, color=colors, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(20, color='red', linewidth=0.8, linestyle='--', alpha=0.5, label='+20% threshold')
ax.axhline(-20, color='red', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_ylabel('Relative Change (%)')
ax.set_title('Gradient Change When State Zeroed')
plt.setp(ax.get_xticklabels(), rotation=20, ha='right', fontsize=9)
ax.legend(fontsize=8)

# --- Plot 3: Loss Comparison ---
ax = axes[2]
conditions = ['Baseline', 'State Ablated']
losses = [loss.item(), ablated_loss.item()]
bar_colors = ['steelblue', 'coral']
ax.bar(conditions, losses, color=bar_colors, alpha=0.8)
ax.set_ylabel('Flow Matching Loss')
ax.set_title('Loss: Baseline vs State Ablated')
for bar, val in zip(ax.patches, losses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()

plot_path = str(GRADIENT_DIR / 'smolvla_gradient_analysis.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f'Saved: {plot_path}')

try:
    from IPython.display import Image as IPImage, display
    display(IPImage(plot_path))
except Exception:
    pass

## 19. Layer-wise Gradient Profile

In [ ]:
# Profile gradient norms layer by layer across the full model

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Re-run baseline forward+backward
policy.zero_grad()
try:
    loss_r, _ = policy.forward(batch)
except Exception:
    loss_r = policy.compute_loss(batch)
    if isinstance(loss_r, tuple): loss_r = loss_r[0]
loss_r.backward()

# Collect per-parameter gradient norms, grouped by module prefix
grad_profile = {}
for name, param in policy.named_parameters():
    if param.grad is not None:
        # Shorten name for display
        parts = name.split('.')
        group = '.'.join(parts[:3]) if len(parts) >= 3 else name
        if group not in grad_profile:
            grad_profile[group] = []
        grad_profile[group].append(param.grad.float().norm().item())

# Aggregate
group_norms = {k: float(np.sqrt(sum(v**2 for v in vals)))
               for k, vals in grad_profile.items()}

# Sort by norm descending
sorted_groups = sorted(group_norms.items(), key=lambda x: -x[1])[:30]  # top 30
names_plot = [n for n, _ in sorted_groups]
norms_plot = [v for _, v in sorted_groups]

# Highlight state_proj
colors = ['red' if 'state' in n.lower() else 'steelblue' for n in names_plot]

fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(range(len(names_plot)), norms_plot, color=colors, alpha=0.8)
ax.set_xticks(range(len(names_plot)))
ax.set_xticklabels(names_plot, rotation=45, ha='right', fontsize=7)
ax.set_yscale('log')
ax.set_ylabel('Gradient Norm (log scale)')
ax.set_title('SmolVLA: Layer-wise Gradient Profile (red = state encoder)')

plt.tight_layout()
profile_path = str(GRADIENT_DIR / 'smolvla_layerwise_gradients.png')
plt.savefig(profile_path, dpi=150, bbox_inches='tight')
print(f'Saved: {profile_path}')

try:
    from IPython.display import Image as IPImage, display
    display(IPImage(profile_path))
except Exception:
    pass

## 20. Export Results

In [ ]:
# Export all results to JSON

import json
from datetime import datetime

export = {
    'model': MODEL_ID,
    'timestamp': datetime.now().isoformat(),
    'gradient_analysis': {
        'baseline_loss': loss.item(),
        'ablated_loss': ablated_loss.item(),
        'loss_increase': ablated_loss.item() - loss.item(),
        'baseline_gradient_norms': baseline_stats,
        'ablated_gradient_norms': ablated_stats,
        'gradient_comparison': comparison,
    },
    'baseline_benchmarks': {
        'libero': LIBERO_RESULTS,
        'metaworld': MW_RESULTS,
        'vlabench': VLABENCH_RESULTS,
    },
    'ablation_benchmarks': {
        'libero': LIBERO_ABLATION,
        'metaworld': MW_ABLATION,
        'vlabench': VLABENCH_ABLATION,
    },
    'model_structure': structure,
}

result_path = RESULTS_DIR / 'smolvla_complete_results.json'
with open(result_path, 'w') as f:
    json.dump(export, f, indent=2, default=str)

print(f'Results saved to: {result_path}')
print(f'\nKey findings:')

# State encoder utilization summary
state_norm_b = baseline_stats.get('state_proj', 0)
vision_norm_b = baseline_stats.get('vision_encoder', 1e-10)
ratio = state_norm_b / (vision_norm_b + 1e-10)
print(f'  state_proj gradient / vision_encoder gradient = {ratio:.4f}')
if ratio < 0.1:
    print('  → State encoder severely underutilized (ratio < 0.1)')
elif ratio < 0.5:
    print('  → State encoder moderately utilized (0.1 < ratio < 0.5)')
else:
    print('  → State encoder well utilized (ratio > 0.5)')

print(f'  Loss increase from state ablation: {ablated_loss.item() - loss.item():.4f}')

---
## 21. Summary

### Research Question
**Does SmolVLA effectively utilize proprioceptive state information?**

SmolVLA maps the robot's joint state to a single VLM token via `state_proj` (Linear),
then concatenates it to the image/language sequence. We measure whether this token
carries task-relevant information or is ignored by the action expert.

### Methodology
1. **Gradient analysis**: Compare `state_proj` gradient norm vs. vision/language encoders
2. **Ablation study**: Zero `state_proj` output → measure success rate drop
3. **Loss sensitivity**: Measure flow matching loss increase when state is zeroed

### Key Metrics
| Metric | Interpretation |
|--------|---------------|
| `state_proj` gradient norm | How strongly does the loss signal update the state encoder? |
| Gradient ratio: state/vision | Relative importance of state vs visual input |
| LIBERO/MetaWorld drop (ablated) | Task-level evidence of state utilization |
| VLABench drop (ablated) | Language-conditioned task evidence |
| Loss increase (ablated) | How much does zeroing state hurt flow matching? |

### Comparison with RDT-1B
Both SmolVLA and RDT-1B use a single Linear layer for state encoding.
SmolVLA differs in:
- **Loss**: Flow matching (not diffusion MSE)
- **Backbone**: SmolVLM2 10x smaller than SigLIP+T5-XXL
- **Framework**: LeRobot (standard fine-tuning pipeline)
- **Compactness**: 450M vs 1.2B parameters

In [ ]:
# Final cleanup
hook_manager.cleanup()
print('All hooks removed')
print('SmolVLA comprehensive study complete.')